In [1]:
!gcloud auth list

    Credentialed Accounts
ACTIVE  ACCOUNT
*       jhaybeecode@gmail.com

To set the active account, run:
    $ gcloud config set account `ACCOUNT`



In [2]:
!gcloud config list

[core]
account = jhaybeecode@gmail.com
disable_usage_reporting = False
project = analytic-sage-data-portal

Your active configuration is: [default]


In [3]:
!gcloud storage buckets create gs://jaybeecode-week3-assignment --location=EU

Creating gs://jaybeecode-week3-assignment/...


In [4]:
!bq mk --dataset --location=EU analytic-sage-data-portal:raw_assign

Dataset 'analytic-sage-data-portal:raw_assign' successfully created.


The free public RPCs kept rate-limiting the second block range. The pipeline treats any error containing "limit" as "range too large", so it kept splitting until it crashed. I switched to my own Infura endpoint and used `--chunk-size 250`, so each finished chunk was checkpointed and re-runs continued where they stopped.

In [ ]:
%env WEB3_RPC_URL=https://mainnet.infura.io/v3/YOUR_KEY

env: WEB3_RPC_URL=https://mainnet.infura.io/v3/d953e7bda11d4ac4941b918b4eb84526


In [5]:
!uv run pipeline.py extract --bucket jaybeecode-week3-assignment --from-block 19000000 --to-block 19002000

  Bucket gs://jaybeecode-week3-assignment exists (location: EU).
2026-11-16 15:42:16,196 [INFO] 1/1 chunks remain
2026-11-16 15:42:16,196 [INFO] chunk 19000000_19002000 starting
2026-11-16 15:42:19,364 [WARNING] get_logs 19000000..19000499 attempt 1 failed: 400 Client Error: Bad Request for url: https://eth.drpc.org/
2026-11-16 15:42:22,331 [WARNING] Span 19000000..19000499 too large - splitting
2026-11-16 15:42:23,277 [WARNING] Span 19000000..19000249 too large - splitting
2026-11-16 15:42:23,668 [WARNING] Span 19000000..19000124 too large - splitting
2026-11-16 15:42:24,043 [WARNING] Span 19000000..19000062 too large - splitting
2026-11-16 15:44:19,780 [WARNING] Span 19000063..19000124 too large - splitting
2026-11-16 15:44:20,145 [WARNING] get_logs 19000063..19000093 attempt 1 failed: {'code': -32000, 'message': 'method not available'}
2026-11-16 15:44:25,238 [WARNING] get_logs 19000063..19000093 attempt 2 failed: 403 Client Error: Forbidden for url: https://ethereum-rpc.publicnode.

In [6]:
!gcloud storage ls -r gs://jaybeecode-week3-assignment/raw/erc20_transfers/

gs://jaybeecode-week3-assignment/raw/erc20_transfers/:

gs://jaybeecode-week3-assignment/raw/erc20_transfers/dt=2024-01-13/:
gs://jaybeecode-week3-assignment/raw/erc20_transfers/dt=2024-01-13/blocks_19000000_19002000.parquet

gs://jaybeecode-week3-assignment/raw/erc20_transfers/dt=2024-01-14/:
gs://jaybeecode-week3-assignment/raw/erc20_transfers/dt=2024-01-14/blocks_19000000_19002000.parquet


In [7]:
!uv run pipeline.py external-table --bucket jaybeecode-week3-assignment --dataset raw_assign --bq-project analytic-sage-data-portal

2026-11-16 21:01:31,006 [INFO] Dataset analytic-sage-data-portal.raw_assign already exists
2026-11-16 21:01:33,017 [INFO] Created external table analytic-sage-data-portal.raw_assign.erc20_transfers_ext over gs://jaybeecode-week3-assignment/raw/erc20_transfers/*


In [8]:
!uv run pipeline.py query --bucket jaybeecode-week3-assignment --dataset raw_assign --dt 2024-01-13 --bq-project analytic-sage-data-portal

Running query:
SELECT dt, COUNT(*) AS transfers FROM `analytic-sage-data-portal.raw_assign.erc20_transfers_ext` WHERE dt = "2024-01-13" GROUP BY dt ORDER BY dt

Row((datetime.date(2024, 1, 13), 192111), {'dt': 0, 'transfers': 1})


In [9]:
!uv run pipeline.py query --bucket jaybeecode-week3-assignment --dataset raw_assign --dt 2024-01-14 --bq-project analytic-sage-data-portal

Running query:
SELECT dt, COUNT(*) AS transfers FROM `analytic-sage-data-portal.raw_assign.erc20_transfers_ext` WHERE dt = "2024-01-14" GROUP BY dt ORDER BY dt

Row((datetime.date(2024, 1, 14), 81864), {'dt': 0, 'transfers': 1})


In [18]:
!uv run pipeline.py extract --bucket jaybeecode-week3-assignment --from-block 19002000 --to-block 19004000 --chunk-size 250

  Bucket gs://jaybeecode-week3-assignment exists (location: EU).
2026-11-16 22:39:21,987 [INFO] 1/8 chunks remain
2026-11-16 22:39:21,987 [INFO] chunk 19003750_19004000 starting
2026-11-16 22:39:25,990 [WARNING] Span 19003750..19003999 too large - splitting
2026-11-16 22:39:42,526 [INFO] chunk 19003750_19004000: wrote 18302 rows to gs://jaybeecode-week3-assignment/raw/erc20_transfers/dt=2024-01-14/blocks_19003750_19004000.parquet
2026-11-16 22:39:42,825 [INFO] progress 1/1 | elapsed 21s | ETA ~0s
2026-11-16 22:39:42,825 [INFO] Backfill complete.


In [19]:
!gcloud storage ls -r gs://jaybeecode-week3-assignment/raw/erc20_transfers/

gs://jaybeecode-week3-assignment/raw/erc20_transfers/:

gs://jaybeecode-week3-assignment/raw/erc20_transfers/dt=2024-01-13/:
gs://jaybeecode-week3-assignment/raw/erc20_transfers/dt=2024-01-13/blocks_19000000_19002000.parquet

gs://jaybeecode-week3-assignment/raw/erc20_transfers/dt=2024-01-14/:
gs://jaybeecode-week3-assignment/raw/erc20_transfers/dt=2024-01-14/blocks_19000000_19002000.parquet
gs://jaybeecode-week3-assignment/raw/erc20_transfers/dt=2024-01-14/blocks_19002000_19002250.parquet
gs://jaybeecode-week3-assignment/raw/erc20_transfers/dt=2024-01-14/blocks_19002250_19002500.parquet
gs://jaybeecode-week3-assignment/raw/erc20_transfers/dt=2024-01-14/blocks_19002500_19002750.parquet
gs://jaybeecode-week3-assignment/raw/erc20_transfers/dt=2024-01-14/blocks_19002750_19003000.parquet
gs://jaybeecode-week3-assignment/raw/erc20_transfers/dt=2024-01-14/blocks_19003000_19003250.parquet
gs://jaybeecode-week3-assignment/raw/erc20_transfers/dt=2024-01-14/blocks_19003250_19003500.parquet
gs://

In [20]:
!uv run pipeline.py query --bucket jaybeecode-week3-assignment --dataset raw_assign --dt 2024-01-14 --bq-project analytic-sage-data-portal

Running query:
SELECT dt, COUNT(*) AS transfers FROM `analytic-sage-data-portal.raw_assign.erc20_transfers_ext` WHERE dt = "2024-01-14" GROUP BY dt ORDER BY dt

Row((datetime.date(2024, 1, 14), 302949), {'dt': 0, 'transfers': 1})


In [21]:
!bq query --use_legacy_sql=false "SELECT COUNT(*) FROM \`analytic-sage-data-portal.raw_assign.erc20_transfers_ext\`"

Error in query string: Error processing job 'analytic-sage-data-
portal:bqjob_r3a3914aaeb06340b_000001a1e662b194_1': Cannot query over table
'analytic-sage-data-portal.raw_assign.erc20_transfers_ext' without a filter over
column(s) 'dt' that can be used for partition elimination


**What happens if you run the query without `WHERE dt = ...`?**

BigQuery refuses to run it. The external table was created with "require partition filter" turned on, so every query has to filter on `dt`.

The rule is there to protect cost. The data in GCS is split into folders by date (`dt=2024-01-13`, `dt=2024-01-14`). With `WHERE dt = ...`, BigQuery only opens the files in that one folder (partition pruning). Without it, BigQuery would read every Parquet file under the prefix, across every date. On a real pipeline that adds data every day, that query would get slower and more expensive every day.